In [1]:
!pip install sdv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata
import warnings
warnings.filterwarnings("ignore")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU: Tesla T4


In [5]:

save_dir = '/content'
TARGET   = "label_encoded"

# 1. Charger les datasets
df_selected    = pd.read_csv(f'{save_dir}/df_selected.csv')
df_transformed = pd.read_csv(f'{save_dir}/df_transformed.csv')

# 2. Fonction d'augmentation TVAE
def augmenter_tvae(df, target_col=TARGET, n_synthetic=500, epochs=300):
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df)
    metadata.update_column(column_name=target_col, sdtype='categorical')

    tvae = TVAESynthesizer(metadata, epochs=epochs, batch_size=500, cuda=True)
    tvae.fit(df)
    synthetic = tvae.sample(num_rows=n_synthetic)

    df_aug = pd.concat([df, synthetic], ignore_index=True)
    print(f"  {len(df)} -> {len(df_aug)} lignes (+{n_synthetic} synthétiques)")
    return df_aug

# 3. Appliquer TVAE sur les deux datasets
print("Augmentation df_selected...")
df_selected_aug = augmenter_tvae(df_selected)

print("\nAugmentation df_transformed...")
df_transformed_aug = augmenter_tvae(df_transformed)

print("\nTerminé !")

Augmentation df_selected...
  588 -> 1088 lignes (+500 synthétiques)

Augmentation df_transformed...
  588 -> 1088 lignes (+500 synthétiques)

Terminé !


In [6]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold

TARGET = "label_encoded"

def preparer(df):
    X = StandardScaler().fit_transform(df.drop(columns=[TARGET]).values)
    y = LabelEncoder().fit_transform(df[TARGET].values)
    return X, y

X_sel,     y_sel     = preparer(df_selected)
X_sel_aug, y_sel_aug = preparer(df_selected_aug)
X_tra,     y_tra     = preparer(df_transformed)
X_tra_aug, y_tra_aug = preparer(df_transformed_aug)

df_selected_aug.to_csv('/content/df_selected_tvae.csv', index=False)
df_transformed_aug.to_csv('/content/df_transformed_tvae.csv', index=False)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats = {}

def evaluer(nom, model, X_sel, y_sel, X_sel_aug, y_sel_aug,
                        X_tra, y_tra, X_tra_aug, y_tra_aug):

    def scores(X, y):
        preds = cross_val_predict(model, X, y, cv=cv)
        return {
            "Acc" : round(accuracy_score(y, preds), 4),
            "F1"  : round(f1_score(y, preds, average='weighted'), 4),
        }

    s1 = scores(X_sel,     y_sel)
    s2 = scores(X_sel_aug, y_sel_aug)
    s3 = scores(X_tra,     y_tra)
    s4 = scores(X_tra_aug, y_tra_aug)

    print(f"\n{nom}")
    print(f"{'':22} {'sel_avant':>12} {'sel_apres':>12} {'tra_avant':>12} {'tra_apres':>12}")
    print(f"  {'Accuracy':<20} {s1['Acc']:>12} {s2['Acc']:>12} {s3['Acc']:>12} {s4['Acc']:>12}")
    print(f"  {'F1-score':<20} {s1['F1']:>12} {s2['F1']:>12} {s3['F1']:>12} {s4['F1']:>12}")

    return {"sel_avant": s1, "sel_apres": s2, "tra_avant": s3, "tra_apres": s4}

In [10]:
# KNN
model = KNeighborsClassifier(n_neighbors=5)
resultats["KNN"] = evaluer("KNN", model,
    X_sel, y_sel, X_sel_aug, y_sel_aug,
    X_tra, y_tra, X_tra_aug, y_tra_aug)


KNN
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8316       0.7969       0.8435       0.8419
  F1-score                   0.8315        0.797       0.8434       0.8419


In [12]:
# SVM
model = SVC(kernel='rbf', random_state=42)
resultats["SVM"] = evaluer("SVM", model,
    X_sel, y_sel, X_sel_aug, y_sel_aug,
    X_tra, y_tra, X_tra_aug, y_tra_aug)


SVM
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8452       0.8079        0.852       0.8346
  F1-score                    0.845       0.8056       0.8522       0.8339


In [13]:
# Decision Tree
model = DecisionTreeClassifier(random_state=42)
resultats["Decision Tree"] = evaluer("Decision Tree", model,
    X_sel, y_sel, X_sel_aug, y_sel_aug,
    X_tra, y_tra, X_tra_aug, y_tra_aug)


Decision Tree
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8146       0.7592       0.8163       0.8153
  F1-score                   0.8145       0.7591       0.8162       0.8152


In [14]:
# Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
resultats["Logistic Regression"] = evaluer("Logistic Regression", model,
    X_sel, y_sel, X_sel_aug, y_sel_aug,
    X_tra, y_tra, X_tra_aug, y_tra_aug)


Logistic Regression
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8418       0.7996       0.8622         0.83
  F1-score                   0.8419       0.7995       0.8622       0.8299


In [15]:
# Naive Bayes
model = GaussianNB()
resultats["Naive Bayes"] = evaluer("Naive Bayes", model,
    X_sel, y_sel, X_sel_aug, y_sel_aug,
    X_tra, y_tra, X_tra_aug, y_tra_aug)


Naive Bayes
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.7738       0.7491       0.7313       0.7482
  F1-score                   0.7733       0.7463       0.7313       0.7482


In [19]:
class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        dummy_input = torch.randn(1, 1, input_dim)
        x = self.conv1(dummy_input)
        x = self.pool(x)
        self.flat_features = x.view(x.size(0), -1).size(1)

        self.fc1 = nn.Linear(self.flat_features, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [20]:
def evaluer_cnn_h(X_sel, y_sel, X_sel_aug, y_sel_aug,
                  X_tra, y_tra, X_tra_aug, y_tra_aug, epochs=30):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def run(X, y):
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        all_preds, all_true = [], []
        for tr, val in skf.split(X, y):
            X_tr  = torch.FloatTensor(X[tr]).to(device)
            y_tr  = torch.LongTensor(y[tr]).to(device)
            X_val = torch.FloatTensor(X[val]).to(device)
            model = CNN1D(X.shape[1], len(np.unique(y))).to(device)
            opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
            loss  = nn.CrossEntropyLoss()
            loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
            model.train()
            for _ in range(epochs):
                for xb, yb in loader:
                    opt.zero_grad(); loss(model(xb), yb).backward(); opt.step()
            model.eval()
            with torch.no_grad():
                preds = model(X_val).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds); all_true.extend(y[val])
        return {
            "Acc": round(accuracy_score(all_true, all_preds), 4),
            "F1" : round(f1_score(all_true, all_preds, average='weighted'), 4),
        }

    s1 = run(X_sel,     y_sel)
    s2 = run(X_sel_aug, y_sel_aug)
    s3 = run(X_tra,     y_tra)
    s4 = run(X_tra_aug, y_tra_aug)

    print(f"\nCNN 1D")
    print(f"{'':22} {'sel_avant':>12} {'sel_apres':>12} {'tra_avant':>12} {'tra_apres':>12}")
    print(f"  {'Accuracy':<20} {s1['Acc']:>12} {s2['Acc']:>12} {s3['Acc']:>12} {s4['Acc']:>12}")
    print(f"  {'F1-score':<20} {s1['F1']:>12} {s2['F1']:>12} {s3['F1']:>12} {s4['F1']:>12}")

    return {"sel_avant": s1, "sel_apres": s2, "tra_avant": s3, "tra_apres": s4}

resultats["CNN 1D"] = evaluer_cnn_h(
    X_sel, y_sel, X_sel_aug, y_sel_aug,
    X_tra, y_tra, X_tra_aug, y_tra_aug)


CNN 1D
                          sel_avant    sel_apres    tra_avant    tra_apres
  Accuracy                   0.8197       0.8171       0.8707       0.8493
  F1-score                   0.8199       0.8169       0.8704       0.8493


In [21]:
print(f"\n{'TABLEAU RECAPITULATIF — F1-score':^80}")
print(f"{'='*80}")
print(f"{'Modele':<22} {'sel_avant':>10} {'sel_apres':>10} {'delta_sel':>10} {'tra_avant':>10} {'tra_apres':>10} {'delta_tra':>10}")
print(f"{'-'*80}")
for nom, s in resultats.items():
    d_sel = round(s["sel_apres"]["F1"] - s["sel_avant"]["F1"], 4)
    d_tra = round(s["tra_apres"]["F1"] - s["tra_avant"]["F1"], 4)
    print(f"  {nom:<20} {s['sel_avant']['F1']:>10} {s['sel_apres']['F1']:>10} {d_sel:>10} {s['tra_avant']['F1']:>10} {s['tra_apres']['F1']:>10} {d_tra:>10}")
print(f"{'='*80}")

df_result = pd.DataFrame([
    {"Modele": nom,
     "sel_avant": s["sel_avant"]["F1"], "sel_apres": s["sel_apres"]["F1"],
     "delta_sel": round(s["sel_apres"]["F1"] - s["sel_avant"]["F1"], 4),
     "tra_avant": s["tra_avant"]["F1"], "tra_apres": s["tra_apres"]["F1"],
     "delta_tra": round(s["tra_apres"]["F1"] - s["tra_avant"]["F1"], 4)}
    for nom, s in resultats.items()
])
df_result.to_csv('/content/resultats_tvae.csv', index=False)


                        TABLEAU RECAPITULATIF — F1-score                        
Modele                  sel_avant  sel_apres  delta_sel  tra_avant  tra_apres  delta_tra
--------------------------------------------------------------------------------
  KNN                      0.8315      0.797    -0.0345     0.8434     0.8419    -0.0015
  SVM                       0.845     0.8056    -0.0394     0.8522     0.8339    -0.0183
  Decision Tree            0.8145     0.7591    -0.0554     0.8162     0.8152     -0.001
  Logistic Regression      0.8419     0.7995    -0.0424     0.8622     0.8299    -0.0323
  Naive Bayes              0.7733     0.7463     -0.027     0.7313     0.7482     0.0169
  CNN 1D                   0.8199     0.8169     -0.003     0.8704     0.8493    -0.0211
